In [ ]:
!pip install -U polars pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.4/833.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 10.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: polars-runtime-32
    Found existing installation: polars-runtime-32 1.35.2
    Uninstalling polars-runtime-32-1.35.2:
      Successfully uninstalled polars-runtime-32-1.35.2
  Attempting uninstall: polars
    Found existing installation: polars 1.35.2
    Uninstalling polars-1.35.2:
      Successfully uninstalled polars-1.35.2


In [ ]:
import zipfile
import os
import polars as pl
import glob

In [ ]:
'''
!mkdir -p downloads

!wget -O /content/downloads/molinetes_2022.zip "https://data.buenosaires.gob.ar/dataset/subte-viajes-molinetes/resource/51f7cdcf-04dd-40c0-b0b1-32b016d3ab6b/download"

!wget -O /content/downloads/molinetes_2023.zip "https://data.buenosaires.gob.ar/dataset/subte-viajes-molinetes/resource/8d752670-38d0-49f6-b6df-0d9948c5e993/download"

!wget -O /content/downloads/molinetes_2024.zip "https://data.buenosaires.gob.ar/dataset/subte-viajes-molinetes/resource/9faf4137-63c9-4f15-848c-248c166b54ea/download"

!wget -O /content/downloads/molinetes_2025.zip "https://data.buenosaires.gob.ar/dataset/subte-viajes-molinetes/resource/0d689701-9efd-4644-85d0-f6a1504509f1/download"
'''


--2026-06-05 03:09:29--  https://data.buenosaires.gob.ar/dataset/subte-viajes-molinetes/resource/51f7cdcf-04dd-40c0-b0b1-32b016d3ab6b/download
Resolving data.buenosaires.gob.ar (data.buenosaires.gob.ar)... 200.16.89.208
Connecting to data.buenosaires.gob.ar (data.buenosaires.gob.ar)|200.16.89.208|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2022.zip [following]
--2026-06-05 03:09:30--  https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2022.zip
Resolving cdn.buenosaires.gob.ar (cdn.buenosaires.gob.ar)... 200.16.89.97
Connecting to cdn.buenosaires.gob.ar (cdn.buenosaires.gob.ar)|200.16.89.97|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 82535697 (79M) [application/zip]
Saving to: ‘/content/downloads/molinetes_2022.zip’

            /conten  34%[=====>              ]  27.29M   411KB/s    eta 2m

In [ ]:
!mkdir -p csv_temp

!unzip "downloads/*.zip" -d csv_temp

Archive:  downloads/molinetes-2025.zip
   creating: csv_temp/molinetes-2025/
  inflating: csv_temp/molinetes-2025/202501_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202501_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202502_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202502_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202503_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202503_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202504_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202504_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202505_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202505_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv  
  inflating: csv_temp/molinetes-2025/202506_PAX15min-ABC-INCLUY

In [ ]:
def procesar_y_convertir_a_parquet(path_csv, carpeta_destino="data_parquet"):
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)

    df_raw = pl.read_csv(
        path_csv,
        has_header=False,
        separator="\x00",
        quote_char=None,
        new_columns=["linea_sucia"]
    )

    df_limpio = (
        df_raw.select(
            pl.col("linea_sucia")
            .str.strip_chars('"')
            .str.replace_all(r";+$", "")
            .str.split(";")
            # Ahora sí usamos el parámetro que pide tu versión de Polars
            .list.to_struct(fields=[f"col_{i}" for i in range(10)])
        )
        .unnest("linea_sucia")
    )

    nombres_columnas = df_limpio.row(0)
    df_final = df_limpio.slice(1).rename({
        # Mapeamos los nombres temporales a los reales del header
        f"col_{i}": str(nombre) for i, nombre in enumerate(nombres_columnas)
    })

    nombre_archivo = os.path.basename(path_csv).replace(".csv", ".parquet")
    path_salida = os.path.join(carpeta_destino, nombre_archivo)

    df_final.write_parquet(path_salida)
    return path_salida

# Ejecutamos el loop
archivos_csv = glob.glob("/content/csv_temp/*.csv")
for f in archivos_csv:
    try:
        procesar_y_convertir_a_parquet(f)
        print(f"✅ {f} convertido a Parquet.")
    except Exception as e:
        print(f"❌ Error en {f}: {e}")

❌ Error en /content/csv_temp/202202_PAX15min-DEH.csv: invalid utf-8 sequence
✅ /content/csv_temp/202308_PAX15min-DEH.csv convertido a Parquet.
❌ Error en /content/csv_temp/202403_PAX15min-DEH.csv: invalid utf-8 sequence
✅ /content/csv_temp/202307_PAX15min-DEH.csv convertido a Parquet.
❌ Error en /content/csv_temp/202207_PAX15min-ABC.csv: invalid utf-8 sequence
✅ /content/csv_temp/202305_PAX15min-DEH.csv convertido a Parquet.
❌ Error en /content/csv_temp/202202_PAX15min-ABC.csv: invalid utf-8 sequence
✅ /content/csv_temp/202301_PAX15min-ABC.csv convertido a Parquet.
✅ /content/csv_temp/202312_PAX15min-ABC.csv convertido a Parquet.
✅ /content/csv_temp/202312_PAX15min-DEH.csv convertido a Parquet.
❌ Error en /content/csv_temp/202205_PAX15min-ABC.csv: invalid utf-8 sequence
✅ /content/csv_temp/202311_PAX15min-ABC.csv convertido a Parquet.
✅ /content/csv_temp/202303_PAX15min-ABC.csv convertido a Parquet.
✅ /content/csv_temp/202306_PAX15min-DEH.csv convertido a Parquet.
✅ /content/csv_temp/2

In [ ]:
# Escanea todos los archivos al instante sin necesidad de unirlos físicamente
df_master = pl.scan_parquet("data_parquet/*.parquet")

In [ ]:
df_master.head(1000000).collect()

FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,pax_TOTAL
str,str,str,str,str,str,str,str,str,str
"""1/4/2022""","""05:15:00""","""05:30:00""","""LineaD""","""LineaD_Bulnes_N_Turn01""","""Bulnes""","""0""","""0""","""1""","""1"""
"""1/4/2022""","""05:15:00""","""05:30:00""","""LineaD""","""LineaD_Carranza_Turn01""","""Ministro Carranza""","""0""","""0""","""1""","""1"""
"""1/4/2022""","""05:15:00""","""05:30:00""","""LineaD""","""LineaD_CongresoTuc_O_Turn03""","""Congreso de Tucuman""","""4""","""0""","""0""","""4"""
"""1/4/2022""","""05:15:00""","""05:30:00""","""LineaD""","""LineaD_J_Hernandez_Oeste_Turn0…","""Jose Hernandez""","""2""","""0""","""1""","""3"""
"""1/4/2022""","""05:15:00""","""05:30:00""","""LineaE""","""LineaE_Boedo_Turn01""","""Boedo""","""1""","""0""","""0""","""1"""
…,…,…,…,…,…,…,…,…,…
"""6/9/2022""","""17:45:00""","""18:00:00""","""LineaC""","""LineaC_Constitucion_Turn09""","""Constitucion""","""3""","""1""","""1""","""5"""
"""6/9/2022""","""17:45:00""","""18:00:00""","""LineaC""","""LineaC_Moreno_S_Turn02""","""Mariano Moreno""","""21""","""0""","""0""","""21"""
"""6/9/2022""","""17:45:00""","""18:00:00""","""LineaC""","""LineaC_SanJuan_Turn01""","""San Juan""","""32""","""0""","""0""","""32"""


In [ ]:
print(df_master.select(pl.len()))

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SELECT [len()]
  Parquet SCAN [data_parquet/202204_PAX15min-DEH.parquet, ... 48 other sources]
  PROJECT */10 COLUMNS
  ESTIMATED ROWS: 21648200


In [ ]:
df_problema = pl.scan_parquet("data_parquet/202311_PAX15min-ABC.parquet")

In [ ]:
df_problema.head(100).collect()

FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,"pax_TOTAL"""
str,str,str,str,str,str,str,str,str,str
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaB""","""LineaB_Lacroze_O_Turn03""","""Federico Lacroze""","""8""","""0""","""0""","""8"""""
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaB""","""LineaB_Alem_S_Turn04""","""Leandro N. Alem""","""4""","""0""","""0""","""4"""""
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaB""","""LineaB_Alem_S_Turn01""","""Leandro N. Alem""","""18""","""0""","""0""","""18"""""
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaB""","""LineaB_Malabia_S_Turn02""","""Malabia""","""1""","""0""","""0""","""1"""""
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaA""","""LineaA_Congreso_N_Turn01""","""Congreso""","""0""","""0""","""2""","""2"""""
…,…,…,…,…,…,…,…,…,…
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaB""","""LineaB_Lacroze_O_Turn05""","""Federico Lacroze""","""5""","""0""","""1""","""6"""""
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaA""","""LineaA_Acoyte_N_Turn02""","""Acoyte""","""2""","""0""","""0""","""2"""""
"""1/11/2023""","""05:15:00""","""05:30:00""","""LineaB""","""LineaB_Pellegrini_E_Turn06""","""Carlos Pellegrini""","""8""","""0""","""0""","""8"""""
